# Bloque 4: Despliegue e inferencia

Este notebook implementa el bloque 4 del harness. Sirve el modelo `@champion` en modo batch (obligatorio) y documenta la configuracion del endpoint real-time (Free Edition no soporta Model Serving). Incluye estrategia de cuarentena paralela, explicacion de training/serving skew y tabla de predicciones para monitoreo (Bloque 5).

In [0]:
# 1. Inferencia batch: scorear todos los clientes activos con @champion

import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
import pandas as pd
from datetime import datetime

# Configuracion
MODEL_FULL_NAME = "workspace.modelos_prod.repurchase_propensity"
CHAMPION_URI = f"models:/{MODEL_FULL_NAME}@champion"
FEATURES_TABLE = "andina_market_lhcl_prod.features.customer_features"
PREDICTIONS_TABLE = "workspace.modelos_prod.batch_predictions"

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

# Obtener la version del champion actual (v1 = champion del Bloque 3)
all_versions = client.search_model_versions(f"name='{MODEL_FULL_NAME}'")
champion_version = min(int(mv.version) for mv in all_versions)  # v1 es champion
print(f"Champion actual: version {champion_version}")

# Features columns (mismas del Bloque 2)
NUMERIC_FEATURES = ['days_since_last_order', 'orders_count_30d', 'orders_count_90d',
                    'avg_ticket_value_90d', 'payment_failure_rate_90d',
                    'view_product_count_14d', 'search_count_14d', 'web_activity_count_14d',
                    'cart_abandonment_sessions_14d', 'tenure_days']
CATEGORICAL_FEATURES = ['favorite_product_sku', 'customer_segment']
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

# Cargar la ultima foto de features por cliente
latest_features = spark.sql(f"""
    SELECT f.*
    FROM {FEATURES_TABLE} f
    INNER JOIN (
        SELECT customer_id, MAX(feature_timestamp) AS max_ts
        FROM {FEATURES_TABLE}
        GROUP BY customer_id
    ) latest
      ON f.customer_id = latest.customer_id 
      AND f.feature_timestamp = latest.max_ts
""")

print(f"Clientes a scorear: {latest_features.count()}")

# Cargar el modelo por alias (el codigo de inferencia no cambia al mover el alias)
model = mlflow.sklearn.load_model(CHAMPION_URI)

# Scorear en pandas (el modelo incluye el preprocessing pipeline internamente)
features_pd = latest_features.select("customer_id", *FEATURE_COLUMNS).toPandas()
probabilities = model.predict_proba(features_pd[FEATURE_COLUMNS])[:, 1]

# Construir tabla de resultados
now = datetime.utcnow()
results = pd.DataFrame({
    "customer_id": features_pd["customer_id"],
    "propensity": probabilities.astype(float),
    "scored_at": now,
    "model_version": int(champion_version),
    "model_alias": "champion",
    "score_date": now.date(),
})

# Escribir a Unity Catalog (CREATE OR REPLACE para idempotencia en demo)
spark.createDataFrame(results).write.format("delta").mode("overwrite").saveAsTable(PREDICTIONS_TABLE)

print(f"\n✅ Predicciones escritas en {PREDICTIONS_TABLE}: {len(results)} filas")
print(f"\nTop 20 clientes con mayor propension:")
display(spark.sql(f"""
    SELECT customer_id, ROUND(propensity, 4) AS propensity, model_version, model_alias
    FROM {PREDICTIONS_TABLE}
    ORDER BY propensity DESC
    LIMIT 20
"""))

print(f"\nDistribucion de propension:")
display(spark.sql(f"""
    SELECT 
        CASE 
            WHEN propensity >= 0.8 THEN 'Muy alta (>=0.8)'
            WHEN propensity >= 0.6 THEN 'Alta (0.6-0.8)'
            WHEN propensity >= 0.4 THEN 'Media (0.4-0.6)'
            WHEN propensity >= 0.2 THEN 'Baja (0.2-0.4)'
            ELSE 'Muy baja (<0.2)'
        END AS bucket,
        COUNT(*) AS clientes
    FROM {PREDICTIONS_TABLE}
    GROUP BY bucket
    ORDER BY bucket
"""))

In [0]:
# 2. Scoring paralelo champion + challenger (estrategia de cuarentena)
# El usuario decidio: ejecutar ambos modelos en paralelo, observar por un par de dias,
# y promover el challenger solo si da mejores resultados en produccion.

CHALLENGER_URI = f"models:/{MODEL_FULL_NAME}@challenger"
challenger_version = max(int(mv.version) for mv in all_versions)  # ultima version es challenger (Bloque 3)
print(f"Challenger actual: version {challenger_version}")

# Cargar challenger
challenger_model = mlflow.sklearn.load_model(CHALLENGER_URI)

# Scorear con challenger sobre los mismos clientes
probabilities_challenger = challenger_model.predict_proba(features_pd[FEATURE_COLUMNS])[:, 1]

# Tabla comparativa para cuarentena (ambos modelos sobre los mismos clientes)
quarantine_results = pd.DataFrame({
    "customer_id": features_pd["customer_id"],
    "champion_propensity": probabilities.astype(float),
    "champion_version": int(champion_version),
    "challenger_propensity": probabilities_challenger.astype(float),
    "challenger_version": int(challenger_version),
    "scored_at": now,
})

QUARANTINE_TABLE = "workspace.modelos_prod.quarantine_comparison"
spark.createDataFrame(quarantine_results).write.format("delta").mode("overwrite").saveAsTable(QUARANTINE_TABLE)

print(f"✅ Tabla de cuarentena creada: {QUARANTINE_TABLE}")
print(f"   Champion v{champion_version} vs Challenger v{challenger_version} sobre {len(quarantine_results)} clientes")
print(f"\nEstrategia: observar predicciones por 2 dias. Si challenger da mejores resultados, promover moviendo @champion.")
display(spark.sql(f"""
    SELECT 
        customer_id,
        ROUND(champion_propensity, 4) AS champion,
        ROUND(challenger_propensity, 4) AS challenger,
        ROUND(challenger_propensity - champion_propensity, 4) AS diferencia
    FROM {QUARANTINE_TABLE}
    ORDER BY ABS(challenger_propensity - champion_propensity) DESC
    LIMIT 15
"""))

## 3. Configuracion de endpoint real-time (Free Edition: configuracion en codigo)

Databricks Free Edition no soporta Model Serving. El PDF dice: *"Si tu ambiente no lo permite, entrégalo como configuración en código."* Aqui esta la configuracion productiva:

### Configuracion del endpoint (productivo)
```python
# En produccion (Databricks Premium/Enterprise):
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    Endpoint, ServedModelInput, ServedModelInputEntity,
    ServedModelInputSource, ExternalModelInput
)

w = WorkspaceClient()

# Crear endpoint que sirve el modelo por alias
w.serving_endpoints.create(
    name="repurchase-propensity-endpoint",
    config=Endpoint(
        served_models=[{
            "name": "repurchase_propensity",
            "model_name": "workspace.modelos_prod.repurchase_propensity",
            "model_version": "auto",  # sigue el alias @champion
            "scale_to_zero_enabled": True,
            "workload_size": "Small",
        }],
        traffic_config={
            "routes": [
                {"served_model_name": "repurchase_propensity", "traffic_percentage": 100}
            ]
        }
    )
)
```

### Despliegue gradual (cuarentena 90/10 -> 100%)
```python
# Fase 1: cuarentena (90% champion, 10% challenger)
traffic_config = {
    "routes": [
        {"served_model_name": "repurchase_propensity_champion", "traffic_percentage": 90},
        {"served_model_name": "repurchase_propensity_challenger", "traffic_percentage": 10}
    ]
}

# Fase 2: despues de 2 dias con buenos resultados -> 100% al challenger
traffic_config = {
    "routes": [
        {"served_model_name": "repurchase_propensity", "traffic_percentage": 100}
    ]
}
# El alias @champion se mueve a la nueva version (una linea de codigo)
```

### Ejemplo de peticion y respuesta
```json
// Request (la app envia solo el customer_id)
{
    "customer_id": "C00001"
}

// Response (el endpoint resuelve features desde Online Feature Store y devuelve la propension)
{
    "customer_id": "C00001",
    "propensity": 0.73,
    "model_version": 1,
    "model_alias": "champion",
    "scored_at": "2026-10-12T08:00:00Z"
}
```

### Como funciona el lookup de features en tiempo real
1. La app envia `customer_id` al endpoint
2. El endpoint busca las features del cliente desde el Online Feature Store (Feature Store lookup automatico)
3. El modelo (con preprocessing embebido) recibe las features raw y devuelve la propension
4. La respuesta se registra en la inference table automaticamente (en produccion)

## 4. Training/Serving Skew: como evitarlo

### Que es training/serving skew?
Es la diferencia entre las features que el modelo vio en entrenamiento y las features que recibe en inferencia. Si son diferentes, las predicciones son incorrectas.

### Como lo evitamos en este proyecto

| Aspecto | Solucion aplicada |
| --- | --- |
| **Preprocessing** | Todo el preprocessing (imputation, scaling, one-hot encoding) vive **dentro** del Pipeline del modelo. En inferencia, el modelo recibe features raw y las transforma igual que en entrenamiento. |
| **Features** | Las features se calculan en `andina_market_lhcl_prod.features.customer_features` (Feature Store en UC). Tanto entrenamiento como inferencia leen de la misma tabla. |
| **Point-in-time** | En entrenamiento, cada fila ve las features tal como estaban en su fecha de corte (point-in-time correct). En inferencia batch, usamos la ultima foto disponible. |
| **Alias** | El codigo de inferencia siempre carga desde `models:/...@champion`. No hay hardcoding de versiones. |

### Tipos de skew y mitigacion
1. **Feature skew:** las features en produccion se calculan diferente -> *mitigado:* misma tabla UC en train y serve
2. **Preprocessing skew:** el scaling/encoding se aplica diferente -> *mitigado:* Pipeline sklearn embebido en el modelo
3. **Data skew:** la distribucion de los datos cambia (drift) -> *mitigado:* Bloque 5 monitorea drift

## README — Bloque 4: Despliegue e inferencia

### Objetivo
Servir el modelo `@champion` en dos modos: batch (obligatorio) y tiempo real (configuracion en codigo, Free Edition).

### Decisiones tomadas (Decide tu)
1. **Despliegue gradual:** Estrategia de cuarentena paralela. Ambos modelos (champion y challenger) corren en paralelo. Se observan por 2 dias. Si el challenger da mejores resultados, se promueve moviendo el alias. En endpoint productivo: 90/10 -> 100%.
2. **Tabla de predicciones:** `workspace.modelos_prod.batch_predictions` (una tabla, simple y suficiente). Columnas: `customer_id`, `propensity`, `scored_at`, `model_version`, `model_alias`, `score_date`. Tabla de cuarentena: `workspace.modelos_prod.quarantine_comparison`.

### Flujo implementado
1. **Batch scoring:** Carga la ultima foto de features por cliente desde UC, scorea con `models:/...@champion`, escribe en `batch_predictions`.
2. **Cuarentena paralela:** Scorea con ambos modelos (champion + challenger) y escribe comparacion en `quarantine_comparison`.
3. **Real-time (config):** Documentacion del endpoint, despliegue gradual 90/10, ejemplo de peticion/respuesta, lookup de features desde Online Feature Store.
4. **Training/serving skew:** Preprocessing embebido en el Pipeline, features desde la misma tabla UC, point-in-time correcto.

### Evidencias a capturar (manual)
1. Screenshot de la tabla `batch_predictions` en Catalog Explorer (mostrar datos y esquema).
2. Screenshot de la tabla `quarantine_comparison` mostrando champion vs challenger.
3. Screenshot de la distribucion de propension (buckets).
4. Guardar en `/harness_config/evidence/block4/`.

### Que esperamos que domines

**Pregunta 1:** Cuando usar `score_batch` (Feature Store) vs `spark_udf` (cargar modelo por alias)?
- `score_batch` hace el lookup de features automatico. `spark_udf` es mas flexible cuando ya tienes las features. Aqui usamos load_model + predict porque las features ya estan en UC.

**Pregunta 2:** Que es training/serving skew y como se evita?
- Es la diferencia entre features de train y serve. Se evita empaquetando preprocessing en el modelo y leyendo features de la misma tabla UC.

**Pregunta 3:** Como funciona scale-to-zero y cuando duele?
- El endpoint se apaga sin trafico. La primera peticion despues tiene latencia alta (cold start). No ideal para tiempo real critico.

**Pregunta 4:** Como capturan las inference tables las peticiones?
- En produccion, Databricks captura request/response automaticamente en una tabla UC. En batch, escribimos las predicciones manualmente en `batch_predictions`.

**Pregunta 5:** Como se programa el batch como job?
- Lakeflow Jobs programa el notebook para correr diario. El notebook es idempotente (sobrescribe la tabla).